# 125.785 Research and Analytics in Economics and Finance
## Session 6: Panel data II: Fama–MacBeth and two-way fixed effects

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nquynh/125785-research-analytics/blob/main/notebooks/S06_student.ipynb)

**Reading:** Brooks (2019) Ch. 11 and 14.2; Python Guide Ch. 21 and 24  
**Data:** `vn_stock_returns_monthly.csv, vn_corp_panel.csv`

**By the end of this session you can:**
- Run Fama–MacBeth cross-sectional regressions
- Compare Fama–MacBeth with pooled OLS
- Estimate a two-way FE model for firm profitability

*All datasets are simulated for teaching. They mimic the structure of Vietnamese listed firms, banks and markets, but are not real company data.*

### 0. Setup

In [ ]:
# Run this cell first in every session
!pip -q install linearmodels        # statsmodels, pandas, numpy, matplotlib are pre-installed on Colab
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
import statsmodels.formula.api as smf
pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 30)

# Where the course data live (GitHub). If you uploaded the CSV files to Colab, use DATA = ""
DATA = os.environ.get("DATA_DIR", "https://raw.githubusercontent.com/nquynh/125785-research-analytics/main/data/")

### 1. Worked example

**Fama–MacBeth** asks whether firm characteristics are priced in the cross-section of returns. Step 1: each month, regress returns on characteristics (size, book-to-market, beta). Step 2: average the monthly slopes and test whether the average is zero, using the time-series SE of the slopes.

In [ ]:
stk = pd.read_csv(DATA + "vn_stock_returns_monthly.csv")
stk["ex_ret"] = stk["ret"] - stk["rf"]
stk["log_bm"] = np.log(stk["book_to_market"])
# betas estimated over the whole sample (simplification; see Exercise 6.3)
betas = stk.groupby("ticker").apply(lambda g: smf.ols("ex_ret ~ mkt_rf", data=g).fit().params["mkt_rf"]).rename("beta")
stk = stk.merge(betas, left_on="ticker", right_index=True)

# Step 1: monthly cross-sectional regressions
lam = stk.groupby("date").apply(lambda g: smf.ols("ex_ret ~ beta + log_mcap + log_bm", data=g).fit().params)
# Step 2: average slopes and t-statistics
fm = pd.DataFrame({"mean": lam.mean(), "se": lam.std() / np.sqrt(len(lam))})
fm["t"] = fm["mean"] / fm["se"]
fm.round(4)

In [ ]:
# The same with linearmodels
from linearmodels import FamaMacBeth
p = stk.copy(); p["date"] = pd.PeriodIndex(p["date"], freq="M").to_timestamp()
p = p.set_index(["ticker", "date"])
fmb = FamaMacBeth(p["ex_ret"], sm.add_constant(p[["beta", "log_mcap", "log_bm"]])).fit()
print(fmb.summary.tables[1])

### 2. Exercises

#### Exercise 6.1 – Interpretation
Which characteristics are priced? Interpret the size slope in monthly and annual terms. Is the market risk premium (slope on beta) significant? Why might it not be?

In [ ]:
# YOUR CODE HERE


**Your interpretation:** _write 2–3 sentences here_

#### Exercise 6.2 – Pooled OLS vs Fama–MacBeth
Estimate a pooled OLS of `ex_ret` on the same characteristics with SEs clustered by month. Compare the t-statistics with Fama–MacBeth. Why do naive OLS SEs overstate precision?

In [ ]:
# YOUR CODE HERE


**Your interpretation:** _write 2–3 sentences here_

#### Exercise 6.3 – Two-way FE for ROA
Using `vn_corp_panel.csv`, estimate roa_w on leverage, size, sales_growth and tangibility with firm and year fixed effects and firm-clustered SEs. Compare the leverage coefficient with the pooled OLS of Session 3. Why does `tangibility` drop out?

In [ ]:
# YOUR CODE HERE


**Your interpretation:** _write 2–3 sentences here_

---
**Before you leave:** File → Save a copy in Drive, then File → Download → .ipynb. Your notebook *is* your log: keep it with your A1/A2 working files.